In [40]:
%pwd

'c:\\Users\\Danns\\Desktop\\Udemy_GenAI\\Projects\\Medical-ChatBot'

In [41]:
import os
os.chdir('c:/Users/Danns/Desktop/Udemy_GenAI/Projects/Medical-ChatBot/')
%pwd

'c:\\Users\\Danns\\Desktop\\Udemy_GenAI\\Projects\\Medical-ChatBot'

In [42]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [43]:
# Load all PDF from a folder
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob='*.pdf',
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [44]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [ ]:
extracted_docs = load_pdf_files('data')

minimized_docs = filter_to_minimal_docs(extracted_docs)

In [46]:
minimized_docs[100].page_content

'• T cell lymphocytes: 644-2200/mm\n3\n, 60-88% of all lym-\nphocytes.\n• B cell lymphocytes: 82-392/mm\n3\n, 3-20% of all lympho-\ncytes.\n• CD4+ lymphocytes: 500-1200/mm\n3\n, 34-67% of all\nlymphocytes.\nAbnormal results\nThe following results in AIDS tests indicate progres-\nsion of the disease:\n• Percentage of CD4+ lymphocytes: less than 20% of all\nlymphocytes.\n• CD4+ lymphocyte count: less than 200 cells/mm\n3\n.\n• Viral load test: Levels more than 5000 copies/mL.\n• /H9252-2-microglobulin: Levels more than 3.5 mg/dL.\n• P24 antigen: Measurable amounts in blood serum.\nResources\nBOOKS\nAvrameas, Stratis, and Therese Ternynck. “Enzyme Linked\nImmunosorbent Assay (ELISA).” In Encyclopedia of\nImmunology.V ol. 1. Ed. Ivan M. Roitt and Peter J. Delves.\nLondon: Academic Press, 1992.\nBennett, Rebecca, and Erin, Charles A. (Editors). HIV and\nAIDS Testing, Screening, and Confidentiality: Ethics, Law,\nand Social Policy.Oxford, England: Oxford University\nPress, 2001.\nEarly HIV I

In [ ]:
# Splitting text into small chunks to fit in llm's context window
def chunking_docs(docs):
    text_splits = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50,)
    text_chunks = text_splits.split_documents(docs)
    return text_chunks


In [ ]:
text_chunk = chunking_docs(minimized_docs)
print(f"Number of chunks: {len(text_chunk)}")
text_chunk[50].page_content

Number of chunks: 5961


'Teresa Norris, R.N.\nMedical Writer\nUte Park, NM\nLisa Papp, R.N.\nMedical Writer\nCherry Hill, NJ\nPatience Paradox\nMedical Writer\nBainbridge Island, W A\nBarbara J. Pettersen\nGenetic Counselor\nGenetic Counseling of Central\nOregon\nBend, OR\nGenevieve Pham-Kanter, M.S.\nMedical Writer\nChicago, IL\nCollette Placek\nMedical Writer\nWheaton, IL\nBelinda Rowland, Ph.D.\nMedical Writer\nV oorheesville, NY\nAndrea Ruskin, M.D.\nWhittingham Cancer Center\nNorwalk, CT\nLaura Ruth, Ph.D.\nMedical, Science, & Technology'

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

# Text(Chunked Docs) -> Vectors
def download_embeddings():
    '''
    Download the HuggingFace embedding model and return the embeddings object.
    '''
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings
embeddings = download_embeddings()

In [50]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [51]:
embeddings.embed_documents(text_chunk[0].page_content)

[[-0.05895743891596794,
  0.05387328192591667,
  0.008316782303154469,
  0.020680485293269157,
  0.0022009508684277534,
  -0.03443443775177002,
  0.08473175019025803,
  0.10776819288730621,
  0.058957379311323166,
  -0.01663866825401783,
  0.06211822107434273,
  -0.0740492194890976,
  -0.030303195118904114,
  0.03807578235864639,
  -0.004251454956829548,
  -0.007986387237906456,
  0.003145285416394472,
  -0.07220278680324554,
  -0.01501195877790451,
  -0.0713619589805603,
  -0.088174007833004,
  0.045201968401670456,
  0.009071114473044872,
  0.04195645451545715,
  -0.0012928901705890894,
  -0.01484022568911314,
  -0.039426445960998535,
  0.007077474147081375,
  -0.036413609981536865,
  -0.0428992360830307,
  -0.04982227459549904,
  0.009418357163667679,
  -0.02269931696355343,
  0.010238060727715492,
  -0.00445388862863183,
  -0.07801611721515656,
  -0.07069577276706696,
  -0.005806411616504192,
  0.03875808045268059,
  0.012042240239679813,
  -0.053253792226314545,
  -0.0963793545961

In [52]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [53]:
Gemini_api_key = os.getenv('GEMINI_API_KEY')
Pinecone_api_key = os.getenv('PINECONE_API_KEY')

os.environ['GOOGLE_API_KEY'] = Gemini_api_key
os.environ['PINECONE_API_KEY'] = Pinecone_api_key

In [54]:
from pinecone import ServerlessSpec
from pinecone import Pinecone

pc = Pinecone(api_key=Pinecone_api_key)
index_name = 'med-chatbot'

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        spec=ServerlessSpec(cloud='aws', region='us-east-1'),
        dimension=384, # Dimension of the embeddings
        metric='cosine' # Cosine similarity search
    )

index = pc.Index(index_name)

In [55]:
index

In [56]:
from langchain_pinecone import PineconeVectorStore
docSearch = PineconeVectorStore.from_documents(
    documents = text_chunk,
    embedding=embeddings,
    index_name=index_name,
    batch_size=5  # Process documents in batches of 10 to stay under Pinecone's 2MB request limit
)

In [57]:
# Load existing Vectors
from langchain_pinecone import PineconeVectorStore

output = PineconeVectorStore.from_existing_index(
    index_name = index_name,
    embedding = embeddings
)

In [58]:
retriever = output.as_retriever(search_type = 'similarity', search_kwargs={'k':3})

In [59]:
retriever_docs = retriever.invoke('What is piles?')
retriever_docs

[Document(id='93c61b31-5ab2-4ab2-9189-7aaf8a3a1767', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 326.0, 'page_label': '327', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='many diseases and infections.\nFeces—(Also called stool.) The solid waste that is left\nafter food is digested. Feces form in the intestines\nand pass out of the body through the anus.\nFetus—A developing baby inside the womb.\nGout—A disease in which uric acid, a waste prod-\nuct that normally passes out of the body in urine,\ncollects in the joints and the kidneys. This causes\narthritis and kidney stones.\nImmune system —The body’s natural defenses\nagainst disease and infection.'),
 Document(id='7abfdc90-65c9-413e-a888-68dcf6ca653c', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 223.0, 'page_lab


---> 47 from langchain_google_genai.chat_models import ChatGoogleGenerativeAI

     48 from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
     
     49 from langchain_google_genai.llms import GoogleGenerativeAI

In [60]:
#LLM invoke
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [61]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain 
from langchain_core.prompts import ChatPromptTemplate

In [62]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "{input}")
        ]
)

In [63]:
q_and_a_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, q_and_a_chain)

In [64]:
response = rag_chain.invoke({"input": "What is Acromegaly and gigantism"})
print(response['answer'])

Acromegaly is a disorder characterized by the abnormal release of a specific chemical from the pituitary gland in the brain. This leads to increased growth in bone and soft tissue, along with various other disturbances throughout the body.


In [65]:
response = rag_chain.invoke({"input": "How to treat piles?"})
print(response['answer'])

Most hemorrhoids heal without specific treatment if the patient uses stool softeners to alleviate constipation. Antihemorrhoid creams, ointments, and suppositories containing astringents or protectants can help relieve itching, burning, and irritation. In severe cases, enlarged blood vessels may be eliminated by surgery.
